# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/medhu07/flyrankai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Unit of Analysis + Time Window

**Unit of analysis**

One row represents one pseudonymized content item.

Each row contains trailing 90-day search and engagement metrics describing the performance of that content item for a single client.

**Time window**

Most performance metrics summarize the previous 90 days rather than a single date. The dataset is therefore a snapshot dataset rather than a daily event table.

The output of this analysis is a documented understanding of the dataset's structure and limitations before any modeling or feature engineering.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/medhu07/flyrankai.git
%cd flyrankai
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)

df.head()

Cloning into 'flyrankai'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 133 (delta 47), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.86 MiB | 13.57 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/flyrankai
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
print("Duplicate content_ids:", df["content_id"].duplicated().sum())

print("Unique content items:", df["content_id"].nunique())

Duplicate content_ids: 0
Unique content items: 30000


## Field Classification

### Features
These fields are available before making a refresh recommendation and may be used for modeling.

- impressions_90d
- clicks_90d
- ctr
- search_volume
- avg_position (after treating 0 as missing)
- engagement_rate
- scroll_rate
- word_count
- page_age_days

### Label / Proxy

The starter dataset does not contain a true supervised label describing refresh success.

A future label would need to come from post-refresh outcomes or another carefully designed proxy.

### Context

These fields help identify or group records but should not be used as model inputs.

- content_id
- client_id
- content_type

### Excluded

- trend_direction
- trend_pct
- is_declining_label

These are excluded because they are derived from the same trend information and would introduce target leakage if used as model features.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
columns = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "trend_direction",
    "trend_pct"
]

print(df[columns].head())

             content_id          client_id     content_type  impressions_90d  \
0  content_304f48230142  client_f369cb89fc  keyword article             3803   
1  content_a1fb4e703a9e  client_4e07408562  keyword article            15320   
2  content_9aa793d4d895  client_7f2253d7e2  keyword article            12581   
3  content_331d6c4de07b  client_19581e27de  keyword article            11751   
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article            19140   

   clicks_90d   ctr trend_direction  trend_pct  
0          29  0.76            down      -41.4  
1           7  0.05            down      -57.7  
2          11  0.09            down      -60.9  
3          58  0.49          stable      -13.8  
4          24  0.13            down      -34.7  


## 3. Verify it with queries (grain, counts, missing values, windows)

The following queries verify the claims made in this data contract.

They confirm:
- the dataset grain (one row per content item),
- dataset size,
- missing values,
- availability of time-window metrics,
- and the existence of excluded leakage columns.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Dataset shape:", df.shape)

print("Duplicate content_ids:", df["content_id"].duplicated().sum())

print("Unique content items:", df["content_id"].nunique())


Dataset shape: (30000, 44)
Duplicate content_ids: 0
Unique content items: 30000


In [8]:
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

print(missing[missing > 0])

provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
cpc                   2468
competition           2468
main_intent           2374
scroll_rate            125
dtype: int64


In [9]:
time_window_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

df[time_window_cols].head()

,impressions_90d,clicks_90d,sessions_90d,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d
0,3803,29,17,578,2,2,987,13,9
1,15320,7,9,2501,2,3,5915,1,2
2,12581,11,11,2382,1,1,6089,3,3
3,11751,58,78,3626,22,35,4206,17,26
4,19140,24,145,4211,10,14,6452,2,9


In [10]:
df[["trend_direction", "trend_pct"]].head()

,trend_direction,trend_pct
0,down,-41.4
1,down,-57.7
2,down,-60.9
3,stable,-13.8
4,down,-34.7


## 4. Data Limits

This dataset is a snapshot of content performance rather than a daily time-series dataset.

Limitations:

- Metrics are aggregated over trailing windows instead of daily observations.
- No explicit supervised target label is provided.
- trend_direction and trend_pct leak future performance information and should not be used for prediction.
- content_id and client_id identify records but should not be treated as predictive features.
- Results from models trained on this dataset should support decisions rather than establish causal relationships.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])


Rows: 30000
Columns: 44

Missing values per column:
search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.